# Hoja de trabajo 2 — Transformers y mecanismos de atención
**CC3092 Deep Learning y Sistemas Inteligentes** · Fernando Andree Hernández Martínez (23645)

Contenido del notebook:
1. Configuración y semillas
2. Ejercicio a mano de *scaled dot-product attention* y su verificación numérica
3. Verificaciones contra `F.scaled_dot_product_attention`, `nn.MultiheadAttention` y máscaras causales
4. Escalamiento por $\sqrt{d_k}$: varianza de $q\cdot k$, saturación de la softmax y gradientes
5. Costo computacional de la self-attention respecto a $n$
6. Mapas de atención en BERT y GPT-2 (patrones por cabeza)
7. Correferencia de «it» (*tired* vs *wide*)
8. GPT-2: matriz triangular inferior y *attention sinks*
9. Entropía de la atención por capa
10. (Opcional) bertviz

In [ ]:
# Configuración: semillas fijas para reproducibilidad y rutas del proyecto
import sys, math, time, random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
FIG = ROOT / "figures"
FIG.mkdir(exist_ok=True)

from attention import (scaled_dot_product_attention, causal_mask, multi_head_attention,
                       get_attentions, attention_entropy, head_patterns, plot_heads)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_printoptions(precision=4, sci_mode=False)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight"})
print("torch", torch.__version__)

## 2. Ejercicio a mano

Tres tokens con $d_k = d_v = 2$ y, para que las cuentas sean manejables, $W_Q=W_K=W_V=I$, de modo que $Q=K=V=X$:

$$X=\begin{pmatrix}1&0\\0&1\\1&1\end{pmatrix}$$

**Paso 1 — puntajes** $S = QK^\top$:
$S=\begin{pmatrix}1&0&1\\0&1&1\\1&1&2\end{pmatrix}$

**Paso 2 — escalar** por $\sqrt{d_k}=\sqrt2\approx1.4142$:
$S/\sqrt2=\begin{pmatrix}0.7071&0&0.7071\\0&0.7071&0.7071\\0.7071&0.7071&1.4142\end{pmatrix}$

**Paso 3 — softmax por fila.** Fila 1: $e^{0.7071}=2.0281$, $e^0=1$ → suma $5.0562$ → $(0.4011,\,0.1978,\,0.4011)$.
Fila 2 es la permutación de la fila 1: $(0.1978,\,0.4011,\,0.4011)$.
Fila 3: $e^{0.7071}=2.0281$, $e^{1.4142}=4.1133$ → suma $8.1695$ → $(0.2483,\,0.2483,\,0.5035)$.

**Paso 4 — salida** $Z = AV$:
- $z_1 = 0.4011(1,0)+0.1978(0,1)+0.4011(1,1) = (0.8022,\,0.5989)$
- $z_2 = (0.5989,\,0.8022)$
- $z_3 = 0.2483(1,0)+0.2483(0,1)+0.5035(1,1) = (0.7517,\,0.7517)$

Interpretación: el token 3 ($[1,1]$) es el más parecido a todos, por eso recibe más peso en todas las filas; cada salida es un promedio ponderado de los valores.

In [ ]:
# Verificación numérica del ejercicio a mano
X = torch.tensor([[1., 0.], [0., 1.], [1., 1.]])
S = X @ X.T
S_scaled = S / math.sqrt(2)
A_hand = torch.softmax(S_scaled, dim=-1)
Z_hand = A_hand @ X
print("S =\n", S, "\nS/sqrt(2) =\n", S_scaled, "\nA =\n", A_hand, "\nZ = A V =\n", Z_hand)

Z_ref = F.scaled_dot_product_attention(X[None], X[None], X[None])[0]
A_expected = torch.tensor([[0.4011, 0.1978, 0.4011], [0.1978, 0.4011, 0.4011], [0.2483, 0.2483, 0.5035]])
Z_expected = torch.tensor([[0.8022, 0.5989], [0.5989, 0.8022], [0.7517, 0.7517]])
print("A coincide con lo calculado a mano:", torch.allclose(A_hand, A_expected, atol=1e-4))
print("Z coincide con lo calculado a mano:", torch.allclose(Z_hand, Z_expected, atol=1e-4))
print("Z coincide con F.scaled_dot_product_attention:", torch.allclose(Z_hand, Z_ref, atol=1e-6))

## 3. Verificaciones de la implementación

Se compara la implementación propia (`src/attention.py`) con las de PyTorch:
1. `scaled_dot_product_attention` propia vs `F.scaled_dot_product_attention` (sin máscara y con `is_causal=True`).
2. `multi_head_attention` propia vs `nn.MultiheadAttention` copiando sus pesos (`in_proj_weight`, `out_proj.weight`).
3. Propiedades: filas de la atención suman 1, la máscara causal produce una matriz triangular inferior, y `nn.Transformer.generate_square_subsequent_mask` equivale a nuestra máscara booleana.

In [ ]:
B, h, n, d = 2, 4, 7, 16
Q, K, V = torch.randn(B, h, n, d), torch.randn(B, h, n, d), torch.randn(B, h, n, d)

out_mine, w_mine = scaled_dot_product_attention(Q, K, V)
out_torch = F.scaled_dot_product_attention(Q, K, V)
out_mine_c, w_mine_c = scaled_dot_product_attention(Q, K, V, mask=causal_mask(n))
out_torch_c = F.scaled_dot_product_attention(Q, K, V, is_causal=True)

# nn.MultiheadAttention con los mismos pesos que nuestra implementación
d_model, heads = 32, 4
mha = nn.MultiheadAttention(d_model, heads, batch_first=True, bias=False).eval()
x = torch.randn(B, n, d_model)
W_q, W_k, W_v = mha.in_proj_weight.chunk(3)
ours, w_ours = multi_head_attention(x, W_q, W_k, W_v, mha.out_proj.weight, heads)
ref, w_ref = mha(x, x, x, need_weights=True, average_attn_weights=False)

# Máscara de PyTorch: float con -inf arriba de la diagonal y 0 en el resto
torch_mask = nn.Transformer.generate_square_subsequent_mask(n)

checks = {
    "SDPA propia == F.scaled_dot_product_attention": torch.allclose(out_mine, out_torch, atol=1e-5),
    "SDPA causal propia == F.sdpa(is_causal=True)": torch.allclose(out_mine_c, out_torch_c, atol=1e-5),
    "MHA propia == nn.MultiheadAttention (salida)": torch.allclose(ours, ref, atol=1e-5),
    "MHA propia == nn.MultiheadAttention (pesos)": torch.allclose(w_ours, w_ref, atol=1e-5),
    "Filas de atención suman 1": torch.allclose(w_mine.sum(-1), torch.ones(B, h, n)),
    "Atención causal es triangular inferior": bool((w_mine_c.triu(1) == 0).all()),
    "generate_square_subsequent_mask == ~causal_mask": torch.equal(torch.isinf(torch_mask), ~causal_mask(n)),
}
for k, v in checks.items():
    print(f"[{'OK' if v else 'FALLA'}] {k}")
print("Error máximo MHA:", (ours - ref).abs().max().item())